# Rapor: Truncation Blindness

Bu çalışma, dil modellerinin uzun metinlerde kesme (truncation) işlemi sonrası karar verme davranışını incelemektedir. Özellikle önemli bilgilerin metnin sonuna yakın veya sonundan silinmesi durumunda modelin nasıl karar verdiği araştırılmıştır.

## Amaç
- Uzun bir metin kısaltıldığında modelin sonucu değişir mi?
- İstenmeyen bilgi, kesme noktası nedeniyle göz ardı edilir mi?
- Model, metnin başında yer alan anlamı kaybederken yine de güvenli görünür mü?

## Gözlem
Truncation blindness, modelin metin kesildiğinde “görmezden geldiği” veya sonlanan kısma daha fazla ağırlık verdiği durumları ifade eder. Bu etki özellikle çok uzun girdilerde daha belirgin hale gelir. Sonuç olarak, önemli semantik sinyalin metnin sonunda yer alması ya da kesilmesi, sınıflandırma kararını önemli ölçüde etkileyebilir.

## Neden Önemli?
Uzun metinlerde modelin tüm bağlamı aynı anda işleyememesi, belirli bölümlerin göz ardı edilmesine yol açabilir. Bu durum, güvenlik, hakaret, duygu sınıflandırması ve anket yorumlama gibi alanlarda yanlış sonuçlara neden olabilir.

## Sonuç
Truncation blindness, modelin girdi uzunluğuna göre bağlamı “görmezden gelme” eğilimini gösterir. Bu da modelin hem açıklanabilirliği hem de güvenilirliği açısından önemli bir zafiyettir.

## Değerlendirme
- Uzun metinlerde kesim noktaları ve bilgi konumu birlikte değerlendirilmelidir.
- Önemli cümlelerin metnin sonuna yakın yerleştirilmesi, model kararını etkileyebilir.
- Metin uzunluğu arttıkça model çıktılarının stabilitesi kontrol edilmelidir.

Bu rapor, uzun girdilerde kararların sadece son bölüme değil, tüm bağlamın güvenli şekilde işlenmesine bağlı olduğu fikrini vurgular.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Model checkpoints
model_names = {
    "DistilBERT": "distilbert-base-uncased-finetuned-sst-2-english",
    "RoBERTa": "textattack/roberta-base-SST-2",
    "ALBERT": "textattack/albert-base-v2-SST-2"
}

# 1. PRE-LOAD EVERYTHING FIRST
print("Loading all models and tokenizers into memory...")
loaded_models = {}

for name, checkpoint in model_names.items():
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
    loaded_models[name] = (tokenizer, model)

print("All models loaded successfully!\n")

Loading all models and tokenizers into memory...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: textattack/roberta-base-SST-2
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/27 [00:00<?, ?it/s]

All models loaded successfully!



In [2]:
MAX_LENGTH = 21

text_for_truncation = (
    "I had extremely high expectations for this movie because the acting, cinematography, and soundtrack were stunning! "
    "However, as the story unfolded, the atrocious script, horrific pacing, and utterly embarrassing direction ruined the entire film, "
    "making it a complete waste of time and absolute garbage."
)

print("=" * 65)
print("TRUNCATION BLINDNESS")
print("=" * 65)

# --- TRUNCATION BLINDNESS ---
print("\nTRUNCATION BLINDNESS (Full Text vs Truncated)]")

for model_name, (tokenizer, model) in loaded_models.items():
    model.eval()

    # Kesilmemiş ve kesilmiş girişleri hazırla
    full_inputs = tokenizer(
        text_for_truncation,
        truncation=False,
        return_tensors="pt"
    )

    truncated_inputs = tokenizer(
        text_for_truncation,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    # Modele verilen gerçek metin parçalarını göster
    full_text = tokenizer.decode(
        full_inputs["input_ids"][0],
        skip_special_tokens=True
    )

    truncated_text = tokenizer.decode(
        truncated_inputs["input_ids"][0],
        skip_special_tokens=True
    )

    # Her iki giriş için tahmin al
    with torch.no_grad():
        full_output = model(**full_inputs)
        truncated_output = model(**truncated_inputs)

    full_probs = torch.softmax(full_output.logits, dim=-1)[0]
    truncated_probs = torch.softmax(
        truncated_output.logits, dim=-1
    )[0]

    labels = model.config.id2label

    full_id = full_probs.argmax().item()
    truncated_id = truncated_probs.argmax().item()

    print("\n" + "=" * 55)
    print(f"MODEL: {model_name}")
    print("=" * 55)

    print("\nKESİLMEMİŞ METİN")
    print(full_text)
    print(f"Token sayısı: {full_inputs['input_ids'].shape[1]}")
    print("Sınıf olasılıkları:")
    for i, probability in enumerate(full_probs):
        print(f"  {labels[i]}: {probability.item() * 100:.2f}%")
    print(f"En yüksek tahmin: {labels[full_id]}")

    print("\nKESİLMİŞ METİN — MODELE GİREN KISIM")
    print(truncated_text)
    print(f"Token sayısı: {truncated_inputs['input_ids'].shape[1]}")
    print("Sınıf olasılıkları:")
    for i, probability in enumerate(truncated_probs):
        print(f"  {labels[i]}: {probability.item() * 100:.2f}%")
    print(f"En yüksek tahmin: {labels[truncated_id]}")

    print("\nKARŞILAŞTIRMA")
    if full_id != truncated_id:
        print("Karar değişti: EVET")
    else:
        print("Karar değişti: HAYIR")

TRUNCATION BLINDNESS

TRUNCATION BLINDNESS (Full Text vs Truncated)]

MODEL: DistilBERT

KESİLMEMİŞ METİN
i had extremely high expectations for this movie because the acting, cinematography, and soundtrack were stunning! however, as the story unfolded, the atrocious script, horrific pacing, and utterly embarrassing direction ruined the entire film, making it a complete waste of time and absolute garbage.
Token sayısı: 57
Sınıf olasılıkları:
  NEGATIVE: 99.98%
  POSITIVE: 0.02%
En yüksek tahmin: NEGATIVE

KESİLMİŞ METİN — MODELE GİREN KISIM
i had extremely high expectations for this movie because the acting, cinematography, and soundtrack were stunning!
Token sayısı: 21
Sınıf olasılıkları:
  NEGATIVE: 0.09%
  POSITIVE: 99.91%
En yüksek tahmin: POSITIVE

KARŞILAŞTIRMA
Karar değişti: EVET

MODEL: RoBERTa

KESİLMEMİŞ METİN
I had extremely high expectations for this movie because the acting, cinematography, and soundtrack were stunning! However, as the story unfolded, the atrocious script, 